In [ ]:
!pip install ultralytics

In [ ]:
import os
import yaml
import wandb
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient


In [ ]:
!yolo settings wandb=True

In [ ]:

# ==========================================================
# 3. W&B LOGIN (IMPORTANT)
# ==========================================================
try:
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login()
    
    print("✅ W&B login successful")
except Exception as e:
    print("⚠️ W&B login failed:", e)


In [ ]:


# ==========================================================
# 4. DATASET PATH (YOUR STRUCTURE)
# ==========================================================
DATASET_PATH = "/kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo"

data_yaml = {
    "path": DATASET_PATH,
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "crop",
        1: "weed"
    }
}

yaml_path = "/kaggle/working/data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f)

print("✅ data.yaml created at:", yaml_path)

# ==========================================================
# 5. LOAD YOLO SEGMENTATION MODEL
# ==========================================================
model = YOLO("yolo26s-seg.pt")

# ==========================================================
# 6. TRAINING
# ==========================================================
results = model.train(
    data=yaml_path,

    # -----------------------
    # TRAINING SETTINGS
    # -----------------------
    epochs=150,
    patience=20,
    imgsz=960,
    batch=24,

    optimizer="AdamW",
    lr0=0.005,
    weight_decay=0.0005,

    # -----------------------
    # AUGMENTATION (FIELD ROBUST)
    # -----------------------
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=10,
    translate=0.15,
    scale=0.4,
    perspective=0.0005,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.0,
    mixup=0.0,

    # -----------------------
    # SEGMENTATION SETTINGS
    # -----------------------
    overlap_mask=True,

    # -----------------------
    # OUTPUT
    # -----------------------
    project="/kaggle/working/runs/segment",
    name="phenobench_960_YOLO26s-150ep_run",

    # -----------------------
    # PERFORMANCE
    # -----------------------
    cache=True,
    plots=True,

    # -----------------------
    # DEVICE (multi-GPU optional)
    # -----------------------
    device=[0, 1]
)

# ==========================================================
# 7. DONE
# ==========================================================
print("\n🎉 TRAINING COMPLETED")
print("Saved at:", results.save_dir)